# SECNA on three months of LangChain pull requests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdussere/secna/blob/main/notebook/secna.ipynb)

**Summarise · Embed · Cluster · Name · Aggregate.** Data science finds the structure; the LLM gives it meaning.
The LLM reads clusters, not documents, so the bill follows the number of clusters, not the size of the corpus.

The corpus: every pull request opened in the [`langchain-ai`](https://github.com/langchain-ai) organisation on
GitHub over three months. The outputs saved below come from a real run: 14,947 pull requests, 100 clusters,
7 themes, **about $0.15**.

**To run it yourself** (here or in Colab), you need:
- a **GitHub token** (`GITHUB_TOKEN`), since GitHub's search API needs one; any token with public read access will do;
- an **OpenAI-compatible key** (`OPENAI_API_KEY`, and `OPENAI_BASE_URL` for another endpoint), for N and A only.

In Colab, add them under 🔑 *Secrets*. Running everything takes about 20 minutes on a CPU, most of it E; a GPU
runtime is faster. For a quick try, take one repository and one month (cell *Configuration*).

In [1]:
# Setup: in Colab, install what it lacks and read the keys from its Secrets; elsewhere, from the environment
import os
import sys

if "google.colab" in sys.modules:
    %pip install -q sentence-transformers openai
    from google.colab import userdata
    for key in ("GITHUB_TOKEN", "OPENAI_API_KEY", "OPENAI_BASE_URL"):
        try:
            os.environ.setdefault(key, userdata.get(key))
        except Exception:
            pass
else:   # on your machine: the repository's .env (see env.template), and the GitHub CLI's token if it is installed
    import subprocess
    from dotenv import load_dotenv
    load_dotenv("../.env")
    for openai_name, secna_name in (("OPENAI_API_KEY", "SECNA_API_KEY"), ("OPENAI_BASE_URL", "SECNA_BASE_URL")):
        if not os.getenv(openai_name) and os.getenv(secna_name):
            os.environ[openai_name] = os.environ[secna_name]
    if not os.getenv("GITHUB_TOKEN"):
        try:
            os.environ["GITHUB_TOKEN"] = subprocess.run(["gh", "auth", "token"], capture_output=True, text=True).stdout.strip()
        except FileNotFoundError:
            pass
print("GitHub token:", "yes" if os.getenv("GITHUB_TOKEN") else "MISSING — the extraction needs one",
      "· LLM key:", "yes" if os.getenv("OPENAI_API_KEY") else "missing — S, E and C will run; N and A need one")

GitHub token: yes · LLM key: yes


In [2]:
import json
import logging
import re
from pathlib import Path

logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)   # a macOS note
import pandas as pd
import requests
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from tqdm.auto import tqdm

## Configuration
Change `REPOS` to run it on your own code.

In [3]:
REPOS = ["langchain-ai"]                   # a GitHub organisation, or "owner/repository"
SINCE, UNTIL = "2026-07-23", "2026-10-23"  # three months; for a quick try: one repository, one month
MODEL = "gpt-6-luna"                       # a small LLM, for N and A
llm = OpenAI() if os.getenv("OPENAI_API_KEY") else None   # reads OPENAI_API_KEY (and OPENAI_BASE_URL)

## The corpus
Every pull request, from GitHub's public API, one day at a time (a search returns at most 1,000 results): about
9 minutes for three months of LangChain. Saved to `data/`, so the next runs skip it.

In [4]:
QUERY = """query($q: String!, $after: String) { search(query: $q, type: ISSUE, first: 50, after: $after) {
  pageInfo { hasNextPage endCursor }
  nodes { ... on PullRequest { repository { name } title body createdAt author { login }
                               labels(first: 20) { nodes { name } } } } } }"""


def fetch(source, day):
    q = f"{'repo' if '/' in source else 'org'}:{source} is:pr created:{day}"
    prs, after = [], None
    while True:
        page = requests.post("https://api.github.com/graphql", timeout=60,
                             headers={"Authorization": f"bearer {os.environ['GITHUB_TOKEN']}"},
                             json={"query": QUERY, "variables": {"q": q, "after": after}}).json()["data"]["search"]
        prs += page["nodes"]
        if not page["pageInfo"]["hasNextPage"]:
            return prs
        after = page["pageInfo"]["endCursor"]


CACHE = Path("data") / f"{'_'.join(r.replace('/', '__') for r in REPOS)}_{SINCE}_{UNTIL}.parquet"
if CACHE.exists():   # already downloaded
    df = pd.read_parquet(CACHE)
else:
    days = pd.date_range(SINCE, UNTIL, inclusive="left").strftime("%Y-%m-%d")
    prs = [pr for source in REPOS for day in tqdm(days, desc=", ".join(REPOS), unit="day") for pr in fetch(source, day)]
    df = pd.DataFrame({"title": [pr["title"] for pr in prs],
                       "repo": [pr["repository"]["name"] for pr in prs],
                       "labels": [", ".join(label["name"] for label in pr["labels"]["nodes"]) for pr in prs],
                       "description": [pr["body"] or "" for pr in prs],
                       "author": [(pr["author"] or {}).get("login") for pr in prs],
                       "month": [pr["createdAt"][:7] for pr in prs]})
    CACHE.parent.mkdir(exist_ok=True)
    df.to_parquet(CACHE)
print(f"{len(df):,} pull requests · {df['repo'].nunique()} repositories · {df['author'].nunique():,} authors")

langchain-ai:   0%|          | 0/92 [00:00<?, ?day/s]

KeyboardInterrupt: 

## S · Summarise
Clio has a small LLM summarise each conversation: that is 92% of its bill. A pull request is already short and
focused, so here S is plain code: title, repository, labels and description in one short text. **Cost: nothing.**

In [ ]:
df["text"] = ("# " + df["title"] + "\nrepository: " + df["repo"]
              + "\nlabels: " + df["labels"] + "\n\n" + df["description"])

## E · Embed
Each text becomes a point: two texts that say the same thing land side by side. A small open model, on the CPU.
No API key, no bill.

In [ ]:
model = SentenceTransformer("sentence-transformers/LaBSE")
vectors = model.encode(df["text"].tolist(), show_progress_bar=True)
print(f"{len(vectors):,} vectors of {vectors.shape[1]} dimensions")

## C · Cluster
The dense regions, by pure computation: a k-means into 100 clusters. Enough to see the details, few enough to read
them all. No LLM.

In [ ]:
kmeans = KMeans(n_clusters=100, random_state=0).fit(vectors)
df["cluster"] = kmeans.labels_
df["distance"] = kmeans.transform(vectors).min(axis=1)   # to the centre of its cluster
sizes = df["cluster"].value_counts()
print(len(sizes), "clusters · sizes", sizes.min(), "to", sizes.max())

## N · Name
The first LLM call: one per cluster. The model reads the ten pull requests closest to the centre and names the
cluster. It reads ten, it names hundreds.

In [ ]:
tokens = 0


def ask(prompt):
    global tokens
    assert llm, "N and A call an LLM: set OPENAI_API_KEY (and OPENAI_BASE_URL for another endpoint), then rerun the cells from Configuration"
    r = llm.chat.completions.create(model=MODEL, response_format={"type": "json_object"},
                                    messages=[{"role": "user", "content": prompt}])
    tokens += r.usage.total_tokens
    return json.loads(r.choices[0].message.content)


PROMPT = """Here are the most typical pull requests of one cluster. Answer in JSON only:
{"name": "8 to 15 words", "description": "what they change, where and why, in 50 words at most"}

"""
names = {}
for c, members in df.groupby("cluster"):
    typical = members.nsmallest(10, "distance")["text"].str[:3000]
    names[c] = ask(PROMPT + "\n\n".join(typical))
    if (c + 1) % 10 == 0:
        print(f"{c + 1}/100 · {names[c]['name']}")
df["cluster_name"] = df["cluster"].map(lambda c: names[c]["name"])
print(f"{len(names)} clusters named · {tokens:,} tokens")

## A · Aggregate
One single call: the model reads the hundred names and gathers them into a few themes.

In [ ]:
PROMPT = """Here are the clusters found in those pull requests. Gather them into 5 to 7 themes, lettered A, B, C…
in order of importance; every cluster belongs to exactly one theme. Answer in JSON only:
{"themes": [{"id": "A", "name": "3 to 8 words", "clusters": "comma-separated cluster numbers"}, ...]}

"""
themes = ask(PROMPT + "\n".join(f"{c}: {n['name']}" for c, n in names.items()))["themes"]
theme = {int(c): f"{t['id']}  {t['name']}" for t in themes for c in re.findall(r"\d+", str(t["clusters"]))}
df["theme"] = df["cluster"].map(theme).fillna("?  unassigned")
print(df["theme"].value_counts().sort_index().to_string())

## The picture
The themes by size, split by repository.

In [ ]:
import matplotlib.pyplot as plt

top = df["repo"].value_counts().index[:6]
counts = pd.crosstab(df["theme"], df["repo"].where(df["repo"].isin(top), "other"))
counts = counts[[*top, "other"]].loc[counts.sum(axis=1).sort_values().index]
ax = counts.plot.barh(stacked=True, figsize=(11, 6), width=0.6, edgecolor="white")
for y, total in enumerate(counts.sum(axis=1)):
    ax.text(total + 30, y, f"{total:,}  ({total / len(df):.0%})", va="center")
ax.set_title(f"What happened in {', '.join(REPOS)}: {len(df):,} pull requests, {SINCE} → {UNTIL}", loc="left")
ax.spines[:].set_visible(False)
ax.set_xticks([])
ax.set_ylabel("")
plt.show()

## The unexpected
A map shows what you didn't know to look for. The largest cluster, and how it grew month by month.

In [ ]:
c = sizes.index[0]
print(f"cluster {c} · {sizes[c]} pull requests ({sizes[c] / len(df):.0%}) · {names[c]['name']}")
print(names[c]["description"])
print(df[df["cluster"] == c]["month"].value_counts().sort_index().to_string())

## The bill
S cost nothing, E ran on the CPU, C is computation: the only bill is N and A, about a hundred calls.

In [ ]:
print(f"N + A: {tokens:,} tokens for {len(df):,} pull requests (with gpt-6-luna in October 2026: about $0.15)")

---
*SECNA* is the name the talk gives the recipe; the recipe is [Clio's](https://arxiv.org/abs/2412.13678) and
others'. The presentation, the sources and the full test-driven version of this notebook:
[github.com/mdussere/secna](https://github.com/mdussere/secna).